# Week #5(b): Building Transformation Pipelines

[Open this notebook in Colab](https://colab.research.google.com/github/neurosynthetic/grant-vogt/blob/main/notebooks/week05b.ipynb)

> **Important:** Before you do any work, save your own copy to Google Drive.
>
> In Colab: **File → Save a copy in Drive**

You have combined functions before. In Week 2, you recolored a grid and then flipped it. In Weeks 4 and 5(a), you connected several helper functions to build complete transformations.

This week, **composition becomes the main idea**. A library of transformations is already provided. Your job is to study each mystery, choose the right library functions, and connect them into a **pipeline**:

> **input → transformation → transformation → output**

You are not implementing the library. You are practicing how small, reusable functions can be composed to solve larger problems.

## Goals

By the end of this notebook, you will be able to:

- read a function's name, inputs, and description
- select an appropriate transformation from a library
- combine several transformations into a pipeline
- use intermediate variables to make a pipeline easier to understand
- recognize that changing the order of functions can change the result
- solve ten independent ARC-style mysteries

## Setup

Run the next two cells once:

1. **Setup utilities** provides grid drawings and checker buttons.
2. **Transformation library** provides all ten functions used this week.

Both cells are collapsed because you only need to call the functions, not study their implementations.

In [ ]:
#@title Setup utilities (run once)

import copy
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from matplotlib.colors import ListedColormap

GRID_COLORS = [
    "#000000", "#0074D9", "#FF4136", "#2ECC40", "#FFDC00",
    "#AAAAAA", "#F012BE", "#FF851B", "#7FDBFF", "#870C25"
]
GRID_CMAP = ListedColormap(GRID_COLORS)


def draw_grid(ax, grid, title=""):
    rows = len(grid)
    cols = len(grid[0]) if rows else 0
    ax.imshow(grid, cmap=GRID_CMAP, vmin=0, vmax=9)

    font_size = max(6, min(12, 120 // max(rows, cols, 1)))
    colors_with_white_text = {0, 1, 2, 6, 9}
    for row_index, row in enumerate(grid):
        for column_index, color in enumerate(row):
            text_color = "white" if color in colors_with_white_text else "black"
            ax.text(
                column_index, row_index, str(color),
                ha="center", va="center",
                color=text_color, fontsize=font_size, fontweight="bold",
            )

    ax.set_title(title)
    ax.set_xticks([x - 0.5 for x in range(1, cols)], minor=True)
    ax.set_yticks([y - 0.5 for y in range(1, rows)], minor=True)
    ax.grid(which="minor", color="white", linewidth=1)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)


def show_grid(grid, title="Grid"):
    _, ax = plt.subplots(figsize=(4, 3))
    draw_grid(ax, grid, title)
    plt.show()


def show_pair(input_grid, output_grid, label=""):
    _, axes = plt.subplots(1, 2, figsize=(8, 3))
    draw_grid(axes[0], input_grid, "Input")
    draw_grid(axes[1], output_grid, "Output")
    if label:
        plt.suptitle(label)
    plt.tight_layout()
    plt.show()


def show_examples(examples):
    for number, example in enumerate(examples, start=1):
        show_pair(example["input"], example["output"], f"Training example {number}")


def make_check_button(check_fn):
    button = widgets.Button(description="Check Exercise", button_style="info")
    output = widgets.Output()

    def on_click(_):
        output.clear_output()
        with output:
            check_fn()

    button.on_click(on_click)
    display(button, output)


print("Setup complete.")

In [ ]:
#@title Transformation library (provided — run once)

def replace_color(grid, old_color, new_color):
    new_grid = []
    for row in grid:
        new_row = []
        for cell in row:
            if cell == old_color:
                new_row.append(new_color)
            else:
                new_row.append(cell)
        new_grid.append(new_row)
    return new_grid


def flip_horizontal(grid):
    new_grid = []
    for row in grid:
        new_grid.append(row[::-1])
    return new_grid


def flip_vertical(grid):
    new_grid = []
    for row in grid[::-1]:
        new_grid.append(row[:])
    return new_grid


def transpose_grid(grid):
    new_grid = []
    width = len(grid[0])
    for column_index in range(width):
        new_row = []
        for row in grid:
            new_row.append(row[column_index])
        new_grid.append(new_row)
    return new_grid


def crop_object(grid):
    occupied_rows = []
    occupied_columns = []

    for row_index in range(len(grid)):
        for column_index in range(len(grid[0])):
            if grid[row_index][column_index] != 0:
                occupied_rows.append(row_index)
                occupied_columns.append(column_index)

    top = occupied_rows[0]
    bottom = occupied_rows[0]
    for row_index in occupied_rows:
        if row_index < top:
            top = row_index
        if row_index > bottom:
            bottom = row_index

    left = occupied_columns[0]
    right = occupied_columns[0]
    for column_index in occupied_columns:
        if column_index < left:
            left = column_index
        if column_index > right:
            right = column_index

    cropped = []
    for row_index in range(top, bottom + 1):
        new_row = []
        for column_index in range(left, right + 1):
            new_row.append(grid[row_index][column_index])
        cropped.append(new_row)
    return cropped


def add_border(grid, color):
    width = len(grid[0]) + 2
    border_row = [color] * width
    new_grid = [border_row[:]]

    for row in grid:
        new_grid.append([color] + row[:] + [color])

    new_grid.append(border_row[:])
    return new_grid


def remove_border(grid):
    new_grid = []
    for row_index in range(1, len(grid) - 1):
        new_grid.append(grid[row_index][1:-1])
    return new_grid


def scale_grid(grid, factor):
    new_grid = []
    for row in grid:
        expanded_row = []
        for cell in row:
            for repeat in range(factor):
                expanded_row.append(cell)
        for repeat in range(factor):
            new_grid.append(expanded_row[:])
    return new_grid


def tile_horizontal(grid, times):
    new_grid = []
    for row in grid:
        new_row = []
        for repeat in range(times):
            new_row = new_row + row
        new_grid.append(new_row)
    return new_grid


def tile_vertical(grid, times):
    new_grid = []
    for repeat in range(times):
        for row in grid:
            new_grid.append(row[:])
    return new_grid


print("Transformation library ready.")

## Transformation library menu

You may call any of these functions. Every function returns a **new grid** and leaves its input unchanged.

| Function | What it returns |
| --- | --- |
| `replace_color(grid, old_color, new_color)` | Replaces one color everywhere |
| `flip_horizontal(grid)` | Reverses the cells within every row |
| `flip_vertical(grid)` | Reverses the order of the rows |
| `transpose_grid(grid)` | Turns rows into columns |
| `crop_object(grid)` | Removes empty black space around the nonzero object |
| `add_border(grid, color)` | Adds a one-cell border |
| `remove_border(grid)` | Removes a one-cell outer border |
| `scale_grid(grid, factor)` | Makes every cell larger in both directions |
| `tile_horizontal(grid, times)` | Repeats the complete grid from left to right |
| `tile_vertical(grid, times)` | Repeats the complete grid from top to bottom |

## How to build a pipeline

You may use intermediate variables:

```python
def example_pipeline(grid):
    step_1 = crop_object(grid)
    step_2 = replace_color(step_1, 2, 4)
    return step_2
```

Or you may nest function calls:

```python
def example_pipeline(grid):
    return replace_color(crop_object(grid), 2, 4)
```

The first style is often easier to read and debug.

### Order matters

A pipeline runs from the inside outward, or from one intermediate variable to the next. Changing the order may change the result:

```python
add_border(crop_object(grid), 2)
crop_object(add_border(grid, 2))
```

These expressions do different work. Before coding each mystery, write down the steps in the order you think they should happen.

# Core gallery

Complete Mysteries 1–6 first. The first three require one library function. The later mysteries require more than one.

Each mystery is independent. Getting stuck on one does not prevent you from trying another.

# Mystery 1

Study both examples. Identify what changes, what stays the same, and which library function or functions could produce the output.

In [ ]:
mystery_1_examples = [
    {
        "input": [
            [0, 2, 0],
            [2, 2, 0]
        ],
        "output": [
            [0, 4, 0],
            [4, 4, 0]
        ]
    },
    {
        "input": [
            [2, 0],
            [0, 2],
            [2, 2]
        ],
        "output": [
            [4, 0],
            [0, 4],
            [4, 4]
        ]
    }
]

show_examples(mystery_1_examples)

### My pipeline for Mystery 1

- **First:**
- **Then:**
- **Parameters such as colors, factors, or repetition counts:**

In [ ]:
def transform_1(grid):
    # Connect one or more library functions.
    return None

In [ ]:
#@title Check Mystery 1

def check_mystery_1():
    hidden_input = [
    [
        0,
        2,
        2
    ],
    [
        2,
        0,
        0
    ]
]
    hidden_expected = [
    [
        0,
        4,
        4
    ],
    [
        4,
        0,
        0
    ]
]

    try:
        original = copy.deepcopy(hidden_input)
        got = transform_1(hidden_input)
        if got != hidden_expected:
            print("Not yet. Compare the grid size, colors, and cell positions after each step.")
            print("Got:     ", got)
            print("Expected:", hidden_expected)
            return
        if hidden_input != original:
            print("The output is correct, but the input grid changed.")
            return
    except Exception as exc:
        print("Your transformation raised an error:", exc)
        return

    print("Mystery 1 solved: A new color!")

make_check_button(check_mystery_1)

# Mystery 2

Study both examples. Identify what changes, what stays the same, and which library function or functions could produce the output.

In [ ]:
mystery_2_examples = [
    {
        "input": [
            [1, 0, 2, 0],
            [0, 3, 3, 0]
        ],
        "output": [
            [0, 2, 0, 1],
            [0, 3, 3, 0]
        ]
    },
    {
        "input": [
            [4, 0, 0],
            [4, 4, 0],
            [0, 0, 5]
        ],
        "output": [
            [0, 0, 4],
            [0, 4, 4],
            [5, 0, 0]
        ]
    }
]

show_examples(mystery_2_examples)

### My pipeline for Mystery 2

- **First:**
- **Then:**
- **Parameters such as colors, factors, or repetition counts:**

In [ ]:
def transform_2(grid):
    # Connect one or more library functions.
    return None

In [ ]:
#@title Check Mystery 2

def check_mystery_2():
    hidden_input = [
    [
        1,
        2,
        0
    ],
    [
        0,
        3,
        4
    ]
]
    hidden_expected = [
    [
        0,
        2,
        1
    ],
    [
        4,
        3,
        0
    ]
]

    try:
        original = copy.deepcopy(hidden_input)
        got = transform_2(hidden_input)
        if got != hidden_expected:
            print("Not yet. Compare the grid size, colors, and cell positions after each step.")
            print("Got:     ", got)
            print("Expected:", hidden_expected)
            return
        if hidden_input != original:
            print("The output is correct, but the input grid changed.")
            return
    except Exception as exc:
        print("Your transformation raised an error:", exc)
        return

    print("Mystery 2 solved: A mirror!")

make_check_button(check_mystery_2)

# Mystery 3

Study both examples. Identify what changes, what stays the same, and which library function or functions could produce the output.

In [ ]:
mystery_3_examples = [
    {
        "input": [
            [0, 0, 0, 0, 0],
            [0, 7, 7, 0, 0],
            [0, 7, 0, 0, 0],
            [0, 7, 7, 7, 0],
            [0, 0, 0, 0, 0]
        ],
        "output": [
            [7, 7, 0],
            [7, 0, 0],
            [7, 7, 7]
        ]
    },
    {
        "input": [
            [0, 0, 0, 0],
            [0, 0, 6, 0],
            [0, 6, 6, 0],
            [0, 0, 0, 0]
        ],
        "output": [
            [0, 6],
            [6, 6]
        ]
    }
]

show_examples(mystery_3_examples)

### My pipeline for Mystery 3

- **First:**
- **Then:**
- **Parameters such as colors, factors, or repetition counts:**

In [ ]:
def transform_3(grid):
    # Connect one or more library functions.
    return None

In [ ]:
#@title Check Mystery 3

def check_mystery_3():
    hidden_input = [
    [
        0,
        0,
        0,
        0,
        0
    ],
    [
        0,
        0,
        8,
        0,
        0
    ],
    [
        0,
        8,
        8,
        8,
        0
    ],
    [
        0,
        0,
        8,
        0,
        0
    ]
]
    hidden_expected = [
    [
        0,
        8,
        0
    ],
    [
        8,
        8,
        8
    ],
    [
        0,
        8,
        0
    ]
]

    try:
        original = copy.deepcopy(hidden_input)
        got = transform_3(hidden_input)
        if got != hidden_expected:
            print("Not yet. Compare the grid size, colors, and cell positions after each step.")
            print("Got:     ", got)
            print("Expected:", hidden_expected)
            return
        if hidden_input != original:
            print("The output is correct, but the input grid changed.")
            return
    except Exception as exc:
        print("Your transformation raised an error:", exc)
        return

    print("Mystery 3 solved: Remove the empty space!")

make_check_button(check_mystery_3)

# Mystery 4

Study both examples. Identify what changes, what stays the same, and which library function or functions could produce the output.

In [ ]:
mystery_4_examples = [
    {
        "input": [
            [0, 0, 0, 0, 0],
            [0, 2, 0, 0, 0],
            [0, 2, 2, 2, 0],
            [0, 0, 0, 0, 0]
        ],
        "output": [
            [8, 0, 0],
            [8, 8, 8]
        ]
    },
    {
        "input": [
            [0, 0, 0, 0],
            [0, 0, 2, 0],
            [0, 2, 2, 0],
            [0, 0, 0, 0]
        ],
        "output": [
            [0, 8],
            [8, 8]
        ]
    }
]

show_examples(mystery_4_examples)

### My pipeline for Mystery 4

- **First:**
- **Then:**
- **Parameters such as colors, factors, or repetition counts:**

In [ ]:
def transform_4(grid):
    # Connect one or more library functions.
    return None

In [ ]:
#@title Check Mystery 4

def check_mystery_4():
    hidden_input = [
    [
        0,
        0,
        0,
        0,
        0
    ],
    [
        0,
        2,
        2,
        0,
        0
    ],
    [
        0,
        0,
        2,
        0,
        0
    ],
    [
        0,
        0,
        2,
        0,
        0
    ]
]
    hidden_expected = [
    [
        8,
        8
    ],
    [
        0,
        8
    ],
    [
        0,
        8
    ]
]

    try:
        original = copy.deepcopy(hidden_input)
        got = transform_4(hidden_input)
        if got != hidden_expected:
            print("Not yet. Compare the grid size, colors, and cell positions after each step.")
            print("Got:     ", got)
            print("Expected:", hidden_expected)
            return
        if hidden_input != original:
            print("The output is correct, but the input grid changed.")
            return
    except Exception as exc:
        print("Your transformation raised an error:", exc)
        return

    print("Mystery 4 solved: Crop and recolor!")

make_check_button(check_mystery_4)

# Mystery 5

Study both examples. Identify what changes, what stays the same, and which library function or functions could produce the output.

In [ ]:
mystery_5_examples = [
    {
        "input": [
            [1, 0, 2],
            [3, 3, 0]
        ],
        "output": [
            [4, 4, 4, 4, 4],
            [4, 3, 3, 0, 4],
            [4, 1, 0, 2, 4],
            [4, 4, 4, 4, 4]
        ]
    },
    {
        "input": [
            [1, 0],
            [2, 2],
            [0, 3]
        ],
        "output": [
            [4, 4, 4, 4],
            [4, 0, 3, 4],
            [4, 2, 2, 4],
            [4, 1, 0, 4],
            [4, 4, 4, 4]
        ]
    }
]

show_examples(mystery_5_examples)

### My pipeline for Mystery 5

- **First:**
- **Then:**
- **Parameters such as colors, factors, or repetition counts:**

In [ ]:
def transform_5(grid):
    # Connect one or more library functions.
    return None

In [ ]:
#@title Check Mystery 5

def check_mystery_5():
    hidden_input = [
    [
        5,
        0,
        0
    ],
    [
        0,
        6,
        6
    ],
    [
        7,
        0,
        6
    ]
]
    hidden_expected = [
    [
        4,
        4,
        4,
        4,
        4
    ],
    [
        4,
        7,
        0,
        6,
        4
    ],
    [
        4,
        0,
        6,
        6,
        4
    ],
    [
        4,
        5,
        0,
        0,
        4
    ],
    [
        4,
        4,
        4,
        4,
        4
    ]
]

    try:
        original = copy.deepcopy(hidden_input)
        got = transform_5(hidden_input)
        if got != hidden_expected:
            print("Not yet. Compare the grid size, colors, and cell positions after each step.")
            print("Got:     ", got)
            print("Expected:", hidden_expected)
            return
        if hidden_input != original:
            print("The output is correct, but the input grid changed.")
            return
    except Exception as exc:
        print("Your transformation raised an error:", exc)
        return

    print("Mystery 5 solved: Flip and frame!")

make_check_button(check_mystery_5)

# Mystery 6

Study both examples. Identify what changes, what stays the same, and which library function or functions could produce the output.

In [ ]:
mystery_6_examples = [
    {
        "input": [
            [1, 2, 3],
            [4, 5, 6]
        ],
        "output": [
            [4, 1],
            [5, 2],
            [6, 3]
        ]
    },
    {
        "input": [
            [1, 0],
            [2, 3],
            [0, 4]
        ],
        "output": [
            [0, 2, 1],
            [4, 3, 0]
        ]
    }
]

show_examples(mystery_6_examples)

### My pipeline for Mystery 6

- **First:**
- **Then:**
- **Parameters such as colors, factors, or repetition counts:**

In [ ]:
def transform_6(grid):
    # Connect one or more library functions.
    return None

In [ ]:
#@title Check Mystery 6

def check_mystery_6():
    hidden_input = [
    [
        7,
        0,
        8
    ],
    [
        0,
        9,
        0
    ]
]
    hidden_expected = [
    [
        0,
        7
    ],
    [
        9,
        0
    ],
    [
        0,
        8
    ]
]

    try:
        original = copy.deepcopy(hidden_input)
        got = transform_6(hidden_input)
        if got != hidden_expected:
            print("Not yet. Compare the grid size, colors, and cell positions after each step.")
            print("Got:     ", got)
            print("Expected:", hidden_expected)
            return
        if hidden_input != original:
            print("The output is correct, but the input grid changed.")
            return
    except Exception as exc:
        print("Your transformation raised an error:", exc)
        return

    print("Mystery 6 solved: Turn the grid!")

make_check_button(check_mystery_6)

# Challenge gallery

Mysteries 7–10 use longer pipelines. Complete as many as time allows. The same ten library functions are all you need.

# Mystery 7

Study both examples. Identify what changes, what stays the same, and which library function or functions could produce the output.

In [ ]:
mystery_7_examples = [
    {
        "input": [
            [0, 0, 0, 0, 0],
            [0, 2, 0, 0, 0],
            [0, 2, 2, 0, 0],
            [0, 0, 0, 0, 0]
        ],
        "output": [
            [2, 2, 0, 0],
            [2, 2, 0, 0],
            [2, 2, 2, 2],
            [2, 2, 2, 2]
        ]
    },
    {
        "input": [
            [0, 0, 0, 0, 0],
            [0, 0, 3, 3, 0],
            [0, 0, 0, 0, 0]
        ],
        "output": [
            [3, 3, 3, 3],
            [3, 3, 3, 3]
        ]
    }
]

show_examples(mystery_7_examples)

### My pipeline for Mystery 7

- **First:**
- **Then:**
- **Parameters such as colors, factors, or repetition counts:**

In [ ]:
def transform_7(grid):
    # Connect one or more library functions.
    return None

In [ ]:
#@title Check Mystery 7

def check_mystery_7():
    hidden_input = [
    [
        0,
        0,
        0,
        0
    ],
    [
        0,
        5,
        5,
        0
    ],
    [
        0,
        0,
        5,
        0
    ],
    [
        0,
        0,
        0,
        0
    ]
]
    hidden_expected = [
    [
        5,
        5,
        5,
        5
    ],
    [
        5,
        5,
        5,
        5
    ],
    [
        0,
        0,
        5,
        5
    ],
    [
        0,
        0,
        5,
        5
    ]
]

    try:
        original = copy.deepcopy(hidden_input)
        got = transform_7(hidden_input)
        if got != hidden_expected:
            print("Not yet. Compare the grid size, colors, and cell positions after each step.")
            print("Got:     ", got)
            print("Expected:", hidden_expected)
            return
        if hidden_input != original:
            print("The output is correct, but the input grid changed.")
            return
    except Exception as exc:
        print("Your transformation raised an error:", exc)
        return

    print("Mystery 7 solved: Crop and enlarge!")

make_check_button(check_mystery_7)

# Mystery 8

Study both examples. Identify what changes, what stays the same, and which library function or functions could produce the output.

In [ ]:
mystery_8_examples = [
    {
        "input": [
            [1, 0],
            [2, 3]
        ],
        "output": [
            [2, 3, 2, 3],
            [1, 0, 1, 0]
        ]
    },
    {
        "input": [
            [4],
            [0],
            [5]
        ],
        "output": [
            [5, 5],
            [0, 0],
            [4, 4]
        ]
    }
]

show_examples(mystery_8_examples)

### My pipeline for Mystery 8

- **First:**
- **Then:**
- **Parameters such as colors, factors, or repetition counts:**

In [ ]:
def transform_8(grid):
    # Connect one or more library functions.
    return None

In [ ]:
#@title Check Mystery 8

def check_mystery_8():
    hidden_input = [
    [
        1,
        2,
        3
    ],
    [
        4,
        0,
        5
    ]
]
    hidden_expected = [
    [
        4,
        0,
        5,
        4,
        0,
        5
    ],
    [
        1,
        2,
        3,
        1,
        2,
        3
    ]
]

    try:
        original = copy.deepcopy(hidden_input)
        got = transform_8(hidden_input)
        if got != hidden_expected:
            print("Not yet. Compare the grid size, colors, and cell positions after each step.")
            print("Got:     ", got)
            print("Expected:", hidden_expected)
            return
        if hidden_input != original:
            print("The output is correct, but the input grid changed.")
            return
    except Exception as exc:
        print("Your transformation raised an error:", exc)
        return

    print("Mystery 8 solved: Repeat and flip!")

make_check_button(check_mystery_8)

# Mystery 9

Study both examples. Identify what changes, what stays the same, and which library function or functions could produce the output.

In [ ]:
mystery_9_examples = [
    {
        "input": [
            [8, 8, 8, 8, 8],
            [8, 2, 0, 1, 8],
            [8, 0, 2, 2, 8],
            [8, 8, 8, 8, 8]
        ],
        "output": [
            [6, 0],
            [0, 6],
            [1, 6]
        ]
    },
    {
        "input": [
            [4, 4, 4, 4],
            [4, 2, 3, 4],
            [4, 2, 0, 4],
            [4, 0, 2, 4],
            [4, 4, 4, 4]
        ],
        "output": [
            [6, 6, 0],
            [3, 0, 6]
        ]
    }
]

show_examples(mystery_9_examples)

### My pipeline for Mystery 9

- **First:**
- **Then:**
- **Parameters such as colors, factors, or repetition counts:**

In [ ]:
def transform_9(grid):
    # Connect one or more library functions.
    return None

In [ ]:
#@title Check Mystery 9

def check_mystery_9():
    hidden_input = [
    [
        9,
        9,
        9,
        9,
        9
    ],
    [
        9,
        0,
        2,
        0,
        9
    ],
    [
        9,
        2,
        2,
        5,
        9
    ],
    [
        9,
        9,
        9,
        9,
        9
    ]
]
    hidden_expected = [
    [
        0,
        6
    ],
    [
        6,
        6
    ],
    [
        0,
        5
    ]
]

    try:
        original = copy.deepcopy(hidden_input)
        got = transform_9(hidden_input)
        if got != hidden_expected:
            print("Not yet. Compare the grid size, colors, and cell positions after each step.")
            print("Got:     ", got)
            print("Expected:", hidden_expected)
            return
        if hidden_input != original:
            print("The output is correct, but the input grid changed.")
            return
    except Exception as exc:
        print("Your transformation raised an error:", exc)
        return

    print("Mystery 9 solved: Unframe, turn, and recolor!")

make_check_button(check_mystery_9)

# Mystery 10

Study both examples. Identify what changes, what stays the same, and which library function or functions could produce the output.

In [ ]:
mystery_10_examples = [
    {
        "input": [
            [0, 0, 0, 0, 0],
            [0, 3, 0, 0, 0],
            [0, 3, 3, 0, 0],
            [0, 0, 0, 0, 0]
        ],
        "output": [
            [1, 1, 1, 1, 1, 1],
            [1, 7, 7, 0, 0, 1],
            [1, 7, 7, 0, 0, 1],
            [1, 7, 7, 7, 7, 1],
            [1, 7, 7, 7, 7, 1],
            [1, 1, 1, 1, 1, 1]
        ]
    },
    {
        "input": [
            [0, 0, 0],
            [0, 3, 0],
            [0, 0, 0]
        ],
        "output": [
            [1, 1, 1, 1],
            [1, 7, 7, 1],
            [1, 7, 7, 1],
            [1, 1, 1, 1]
        ]
    }
]

show_examples(mystery_10_examples)

### My pipeline for Mystery 10

- **First:**
- **Then:**
- **Parameters such as colors, factors, or repetition counts:**

In [ ]:
def transform_10(grid):
    # Connect one or more library functions.
    return None

In [ ]:
#@title Check Mystery 10

def check_mystery_10():
    hidden_input = [
    [
        0,
        0,
        0,
        0,
        0
    ],
    [
        0,
        0,
        3,
        3,
        0
    ],
    [
        0,
        0,
        3,
        0,
        0
    ],
    [
        0,
        0,
        0,
        0,
        0
    ]
]
    hidden_expected = [
    [
        1,
        1,
        1,
        1,
        1,
        1
    ],
    [
        1,
        7,
        7,
        7,
        7,
        1
    ],
    [
        1,
        7,
        7,
        7,
        7,
        1
    ],
    [
        1,
        7,
        7,
        0,
        0,
        1
    ],
    [
        1,
        7,
        7,
        0,
        0,
        1
    ],
    [
        1,
        1,
        1,
        1,
        1,
        1
    ]
]

    try:
        original = copy.deepcopy(hidden_input)
        got = transform_10(hidden_input)
        if got != hidden_expected:
            print("Not yet. Compare the grid size, colors, and cell positions after each step.")
            print("Got:     ", got)
            print("Expected:", hidden_expected)
            return
        if hidden_input != original:
            print("The output is correct, but the input grid changed.")
            return
    except Exception as exc:
        print("Your transformation raised an error:", exc)
        return

    print("Mystery 10 solved: Build a finished emblem!")

make_check_button(check_mystery_10)

## Composition checkpoint

Look back over your ten pipelines:

- Which mysteries used only one function?
- Which used two, three, or four?
- Did two mysteries use the same function in different positions?
- Where would changing the order produce a different output?
- Which intermediate grid was most helpful to inspect?

### My notes

- **One-function mysteries:**
- **Longest pipeline:**
- **An example where order matters:**
- **Most useful intermediate grid:**

## Optional experiment: Inspect every step

Choose one multi-step mystery. If you used nested function calls, rewrite the pipeline with intermediate variables named `step_1`, `step_2`, and so on. If you already used intermediate variables, keep them.

Then display every intermediate result:

```python
show_grid(step_1, "After step 1")
show_grid(step_2, "After step 2")
```

Compare each grid with what you expected that step to do. This is a useful debugging technique whenever the final output is wrong.

In [ ]:
# Pipeline inspection workspace

# step_1 = ...
# show_grid(step_1, "After step 1")

## Wrap-up and reflection

This week, you solved problems by composing provided functions rather than implementing every operation yourself.

Answer briefly:

- How did function names and descriptions help you choose a transformation?
- When were intermediate variables more helpful than nested calls?
- Give one example where function order mattered.
- Which library function was most reusable?
- How is a long pipeline different from one large function?

### My reflection

- **Choosing functions:**
- **Intermediate variables:**
- **Order mattered when:**
- **Most reusable function:**
- **Pipelines versus one large function:**

## Share your completed notebook

When you finish:

1. Confirm that Mysteries 1–6 pass.
2. Complete as many challenge mysteries as time allows.
3. Complete the reflection.
4. In Colab, click **Share**.
5. Add **michael.d.moffitt@gmail.com** as a Viewer or Commenter.
6. Send the share invite.

The six core mysteries are the main goal. Mysteries 7–10 provide additional composition practice.